# Unit 13 — Conditional 1D U-Net

A temporal U-Net predicts noise for every action feature in the horizon. It shrinks the time axis to gather context, then restores length with skip connections.

**Your result:** the full denoiser network used by Diffusion Policy, assembled from the Unit 12 blocks. Fed the weights of the pinned LeRobot network it gives the same outputs, for the small test network and for the prepared checkpoint. You then change its architecture under a fixed budget and compare.

Full answers are in [the reference notebook](13-diffusion-unet-reference.ipynb).

**Sources.** [Diffusion Policy](https://diffusion-policy.cs.columbia.edu/) (temporal U-Net for actions), the U-Net idea in [DDPM](https://arxiv.org/abs/2006.11239) (appendix B), and the pinned `DiffusionConditionalUnet1d` in `lerobot/policies/diffusion/modeling_diffusion.py` (around line 654; revision `e0d50211ef236143ae867228662b7dfaba554f02`). The pinned model has **no EMA** in the current trainer; the weights you load are the raw trained weights.

**Where this unit sits.** The queue and training objective are available as package functions from Units 10 and 11. Your functions here (`unet_channel_plan`, `unet_forward`, the weight converter) run inside the real network through its hooks.


In [ ]:
import sys
from collections.abc import Sequence
from pathlib import Path
from typing import cast

import torch
from torch import nn

WORKSPACE = next(
    p
    for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (p / "manifests/act_references.yaml").is_file()
)
sys.path.insert(0, str(WORKSPACE / "src"))
from lerobot.configs import FeatureType, PolicyFeature
from lerobot.policies.diffusion import DiffusionConfig
from lerobot.policies.diffusion.modeling_diffusion import DiffusionConditionalUnet1d

import vla_lab.teaching.diffusion as ref
from vla_lab.teaching.blocks import UNetParts
from vla_lab.teaching.diffusion import UNetChannelPlan, unet_from_lerobot

In [ ]:
# Supplied: a helper that the checks use
def close(actual: torch.Tensor, expected: torch.Tensor, message: str, atol: float = 1e-5) -> None:
    """Compare two tensors and say what failed."""
    assert actual.shape == expected.shape, (
        f"{message}: shape {tuple(actual.shape)}, expected {tuple(expected.shape)}"
    )
    worst = (actual.detach() - expected.detach()).abs().max().item()
    assert worst <= atol, f"{message}: largest difference {worst:.2e} is above {atol:.0e}"

## Background: a U-Net over time

The input is an action window `(B, 16, 7)`. The U-Net treats the 16 steps as a 1D signal with 7 channels.

```
encoder:   blocks (7→d0)(d0→d0)  ─skip→  ↓2   blocks (d0→d1)(d1→d1) ─skip→  ↓2   blocks (d1→d2)(d2→d2) ─skip→
middle:    two blocks at width d2
decoder:   [x ‖ skip] (2·d2→d1)(d1→d1) ↑2    [x ‖ skip] (2·d1→d0)(d0→d0) ↑2     head: conv → conv(→7)
```

- Going *down*, the sequence is halved by a stride-2 convolution (all stages except the last) and the channel width grows.
- Each stage keeps a **skip feature**. Going *up*, the newest skip is concatenated on the channel axis, so the decoder blocks take twice the channels.
- Every block is conditioned on `cond = [timestep MLP output, global observation condition]`, the two vectors concatenated.
- In the diagram, $d_0, d_1, d_2$ are the entries of `down_dims`: the channel widths of the encoder stages.
- The decoder has one stage fewer than the encoder: $n-1$ decoder stages for $n$ encoder stages. It upsamples after *every* decoder stage, including the last, with a transposed convolution. With $H=16$ input steps, the length at the bottom of the U-Net and at the output is:

$$H_{\text{bottom}}=\frac{H}{2^{\,n-1}}, \qquad H_{\text{out}}=H_{\text{bottom}}\cdot2^{\,n-1}=H$$

  Where $n$ is the number of encoder stages (the length of `down_dims`) and $H$ the action-window length. $H$ must be divisible by $2^{n-1}$.

**Supplied:** `Conv1dBlock`, the residual block with your Unit 12 function, the module class `ConditionalUnet1D` (it builds itself from your plan function and runs your forward function), and the data reader. **You write** the channel plan and forward pass with skip connections. The weight-name converter is supplied.
### Worked example: shapes through three stages

For `H=16`, action width `A=7`, and widths `(16, 32, 64)`, the encoder carries `(B,7,16) → (B,16,8) → (B,32,4) → (B,64,4)`. It saves a skip at each stage. The decoder uses the last two skips: concatenation doubles channels before each decoder pair, then upsampling returns the time length to 16 and the head returns `(B,16,7)`. This is why skip order and channel counts must agree.

### Why the decoder omits the first pair

With $n$ encoder stages there are $n-1$ downsampling operations, so the decoder needs $n-1$ upsampling stages. Each stage joins a skip at the current reduced length. The highest-resolution skip from the first pair has length $H$; after the final upsample the head already runs at length $H$, so the pinned model has no later decoder block that accepts that skip. The constructor therefore builds decoder entries from `reversed(pairs[1:])`. This is a design choice of this architecture, not a rule for every U-Net.

For four widths (16, 32, 64, 128) and $H=16$, the saved skip lengths are 16, 8, 4, 2. The decoder consumes lengths 2, 4, 8 in that order and then upsamples to 16. It leaves the length-16 skip unused; the head maps the final features to the seven action values.


In [ ]:
# Supplied: a small real pinned U-Net and algorithm-check inputs
IMAGES = ("observation.images.image", "observation.images.image2")


def pinned_unet(
    down_dims: Sequence[int] = (16, 32, 64), cond_dim: int = 6, seed: int = 4
) -> DiffusionConditionalUnet1d:
    """A *real* pinned DiffusionConditionalUnet1d with random weights (7 actions, horizon 16)."""
    torch.manual_seed(seed)
    config = DiffusionConfig(
        input_features={
            IMAGES[0]: PolicyFeature(FeatureType.VISUAL, (3, 64, 64)),
            IMAGES[1]: PolicyFeature(FeatureType.VISUAL, (3, 64, 64)),
            "observation.state": PolicyFeature(FeatureType.STATE, (8,)),
        },
        output_features={"action": PolicyFeature(FeatureType.ACTION, (7,))},
        n_obs_steps=2,
        horizon=16,
        n_action_steps=8,
        drop_n_last_frames=7,
        down_dims=tuple(down_dims),
        diffusion_step_embed_dim=32,
        pretrained_backbone_weights=None,
        push_to_hub=False,
        device="cpu",
    )
    unet = DiffusionConditionalUnet1d(config, global_cond_dim=cond_dim)
    for parameter in unet.parameters():  # distinct biases and scales make wrong wiring visible
        if parameter.dim() == 1:
            nn.init.normal_(parameter, std=0.1)
    return unet.eval()


PINNED = pinned_unet()
X, T_STEPS, COND = (
    torch.randn(3, 16, 7),
    torch.tensor([0, 10, 99]),
    torch.randn(3, 6),
)  # algorithm-check inputs (random numbers)
print("pinned U-Net parameters:", sum(p.numel() for p in PINNED.parameters()))

In [ ]:
PINNED = pinned_unet()
X = torch.randn(2, 16, 7)
T_STEPS = torch.tensor([7, 33])
COND = torch.randn(2, 6)

## Exercise 13.1 — The channel plan

**Purpose.** Decide which block takes how many channels, where the sequence is downsampled, and what the decoder receives after the skip connection is concatenated. A wrong plan builds a network that cannot load the real weights.

**Contract (shapes, units, masks).**

- `unet_channel_plan(action_dim, down_dims)` returns a dict with the keys `"down"`, `"mid"`, `"up"`.
- Let `pairs = [(action_dim, down_dims[0]), (down_dims[0], down_dims[1]), …, (down_dims[-2], down_dims[-1])]`.
- `"down"`: one entry per pair, `(in_channels, out_channels, downsample)`. Each stage has two residual blocks (`in → out`, `out → out`). `downsample` is `True` for every stage **except the last**.
- `"mid"`: `[(c, c), (c, c)]` with `c = down_dims[-1]`.
- `"up"`: one entry per pair **in reverse order, leaving out the first pair** `(action_dim, down_dims[0])`. For a reversed pair `(width_out, width_in)` (that is $(d_{k-1}, d_k)$ read as `(out, in)`, where $d_k$ is `down_dims[k]`) the entry is `(2 * width_in, width_out, True)`: the first block takes the decoder state plus the concatenated skip, hence `2 * width_in`. The pinned network upsamples after every `"up"` stage, so `upsample` is `True` for all of them.

**Method and source.** Read the constructor of the pinned `DiffusionConditionalUnet1d`: the lists `in_out`, `down_modules`, `mid_modules`, `up_modules`. The check builds the real network and compares every channel number and every stride.

**Pointers.** First build `pairs`. Then make `down` from `pairs` with a flag for the last one. Make `up` from `reversed(pairs[1:])`. Print your plan for `(8, 16, 32)`-style inputs and compare with the picture above.

<details><summary>Detailed hint</summary>

```python
pairs = [(action_dim, down_dims[0]), *zip(down_dims[:-1], down_dims[1:])]
down = [(i, o, index < len(pairs) - 1) for index, (i, o) in enumerate(pairs)]
mid = [(down_dims[-1], down_dims[-1])] * 2
up = [(width_in * 2, width_out, True) for width_out, width_in in reversed(pairs[1:])]
return {"down": down, "mid": mid, "up": up}
```
For `down_dims = (16, 32, 64)` and `action_dim = 7`: `pairs = [(7, 16), (16, 32), (32, 64)]`; `up` is `[(128, 32, True), (64, 16, True)]`.

</details>

*This is the reference notebook: the next cell is a complete solution.*

More behaviour checks for this exercise: `check_unet_channel_plan` in [lesson13_checks.py](lesson13_checks.py).

In [ ]:
# Reference solution for Exercise 13.1
def unet_channel_plan(action_dim: int, down_dims: Sequence[int]) -> UNetChannelPlan:
    """Channel plan of the pinned conditional 1D U-Net.

    Returns ``{"down": [...], "mid": [...], "up": [...]}``. Each ``down`` entry is
    ``(in_channels, out_channels, downsample)``: two residual blocks ``in -> out`` and
    ``out -> out``, then a stride-2 convolution unless it is the last stage. ``mid`` is
    ``[(c, c), (c, c)]`` with ``c = down_dims[-1]``. Each ``up`` entry is
    ``(in_channels, out_channels, upsample)`` for the stages in reverse order, skipping the
    first ``(action_dim, down_dims[0])`` pair. Its first block takes ``2 * previous_width``
    channels because the matching encoder feature is concatenated. The pinned network
    upsamples after every ``up`` stage (the last included).
    """
    # Keep this function standalone when copied into notebook solution cells.
    pairs = [(action_dim, down_dims[0]), *zip(down_dims[:-1], down_dims[1:], strict=True)]  # noqa: RUF007
    down = [(i, o, index < len(pairs) - 1) for index, (i, o) in enumerate(pairs)]
    mid = [(down_dims[-1], down_dims[-1])] * 2
    up = [(width_in * 2, width_out, True) for width_out, width_in in reversed(pairs[1:])]
    return {"down": down, "mid": mid, "up": up}

In [ ]:
# Check: Exercise 13.1
for dims in ((16, 32), (16, 32, 64), (8, 16, 24, 32)):
    pinned = pinned_unet(dims)
    plan = unet_channel_plan(7, dims)
    assert set(plan) == {"down", "mid", "up"}, "Return the keys down, mid and up."
    assert (
        len(plan["down"]) == len(pinned.down_modules)
        and len(plan["up"]) == len(pinned.up_modules)
        and len(plan["mid"]) == len(pinned.mid_modules)
    ), f"Wrong number of stages for {dims}."
    for (i, o, down), (res1, res2, downsample) in zip(
        plan["down"], pinned.down_modules, strict=True
    ):
        assert (res1.conv1.block[0].in_channels, res1.conv1.block[0].out_channels) == (i, o), (
            f"down stage channels for {dims}: expected {(res1.conv1.block[0].in_channels, res1.conv1.block[0].out_channels)}, got {(i, o)}"
        )
        assert res2.conv2.block[0].out_channels == o
        assert isinstance(downsample, nn.Conv1d) == bool(down), f"downsample flag for {dims}"
    for (c_in, c_out), block in zip(plan["mid"], pinned.mid_modules, strict=True):
        assert (block.conv1.block[0].in_channels, block.conv1.block[0].out_channels) == (
            c_in,
            c_out,
        )
    for (i, o, up), (res1, res2, upsample) in zip(plan["up"], pinned.up_modules, strict=True):
        assert res2.conv2.block[0].out_channels == o
        assert (res1.conv1.block[0].in_channels, res1.conv1.block[0].out_channels) == (i, o), (
            f"up stage channels for {dims}: expected {(res1.conv1.block[0].in_channels, res1.conv1.block[0].out_channels)}, got {(i, o)}"
        )
        assert isinstance(upsample, nn.ConvTranspose1d) == bool(up), f"upsample flag for {dims}"
print("Exercise 13.1 passed")

# More behaviour checks: lesson13_checks.py, next to this notebook.
from lesson13_checks import check_unet_channel_plan

check_unet_channel_plan(unet_channel_plan)

### Supplied: map prepared weight names

The prepared LeRobot U-Net and the teaching U-Net use different names for five top-level modules. The setup cell supplies a converter that copies each tensor and changes only that prefix. The strict load rejects missing, extra, or wrong-shaped tensors. This keeps the exercise on the model forward path.

### What a skip feature preserves

Each saved skip contains features from before a downsampling step. Downsampling reduces the number of time positions, so some local detail is no longer present in the decoder state. Concatenating the matching skip restores those features at that resolution. The decoder then combines them with its current coarse representation. Save after both residual blocks, and pop in reverse order so the time lengths match.



## Exercise 13.2 — The forward pass with skip connections

**Purpose.** Run the encoder, remember each stage's output, run the middle, then let decoder residual blocks use reduced-resolution skips to restore details lost by downsampling.

**Contract (shapes, units, masks).**

- `unet` has `time_mlp` (timestep → vector), `down_blocks`, `mid_blocks`, `up_blocks`, `head`. `down_blocks` and `up_blocks` hold triples `(res1, res2, resample)`; `res1(x, cond)` and `res2(x, cond)` are residual blocks and `resample(x)` is a downsampling or upsampling module (or `Identity`).
- `x (B, H, A)` (**time before channels**), `timestep (B,)` long, `global_cond (B, G)` or `None`. Return `(B, H, A)`. $B$ is the batch size, $H$ the window length, $A$ the action width, and $G$ the width of the observation condition.
- Move to channels first: `(B, A, H)`. `cond = unet.time_mlp(timestep)`; if `global_cond` is not `None`, concatenate it on the last axis: `cond = [time vector, global_cond]`.
- Encoder: for each triple apply `res1`, then `res2`, **save that output as a skip feature**, then apply `resample`.
- Middle: apply each block of `mid_blocks` to `(x, cond)`.
- Decoder: for each triple, first concatenate `x` with the **most recently saved skip** on the channel axis (`dim=1`), then `res1`, `res2`, then `resample`. Skips are used in reverse order of saving.
- End with `head(x)` and move back to `(B, H, A)`.

**Method and source.** U-Net with skip connections ([Ronneberger et al. 2015](https://arxiv.org/abs/1505.04597), used for denoising in [DDPM](https://arxiv.org/abs/2006.11239) appendix B). Pinned code: `DiffusionConditionalUnet1d.forward`.

**Pointers.** Use a Python list as a stack: `append` after the second residual block of each encoder stage, `pop` in the decoder. Use `torch.cat((x, skip), dim=1)` for channels-first tensors.

<details><summary>Detailed hint</summary>

```python
x = x.transpose(1, 2)                                   # (B, A, H)
cond = unet.time_mlp(timestep)
if global_cond is not None:
    cond = torch.cat([cond, global_cond], dim=-1)
skips = []
for res1, res2, downsample in unet.down_blocks:
    x = res2(res1(x, cond), cond)
    skips.append(x)
    x = downsample(x)
for block in unet.mid_blocks:
    x = block(x, cond)
for res1, res2, upsample in unet.up_blocks:
    x = torch.cat((x, skips.pop()), dim=1)
    x = res2(res1(x, cond), cond)
    x = upsample(x)
return unet.head(x).transpose(1, 2)
```

</details>

*This is the reference notebook: the next cell is a complete solution.*

More behaviour checks for this exercise: `check_unet_forward` in [lesson13_checks.py](lesson13_checks.py).

In [ ]:
# Reference solution for Exercise 13.2
def unet_forward(
    unet: UNetParts,
    x: torch.Tensor,
    timestep: torch.Tensor,
    global_cond: torch.Tensor | None = None,
) -> torch.Tensor:
    """Conditional U-Net forward pass with skip connections.

    ``x (B, H, A)`` (time before channels), ``timestep (B,)``, optional ``global_cond (B, G)``;
    returns ``(B, H, A)``. Steps: move to ``(B, A, H)``; ``cond = time_mlp(timestep)``, with
    ``global_cond`` concatenated on the last axis; for each ``down_blocks`` entry
    ``(res1, res2, downsample)`` apply ``res1(x, cond)``, ``res2(x, cond)``, store the result
    as a skip feature, then ``downsample``; run ``mid_blocks``; for each ``up_blocks`` entry
    ``(res1, res2, upsample)`` concatenate the newest stored skip on the channel axis, apply
    the two blocks, then ``upsample``; finish with ``head`` and return to ``(B, H, A)``.
    """
    x = x.transpose(1, 2)
    cond = unet.time_mlp(timestep)
    if global_cond is not None:
        cond = torch.cat([cond, global_cond], dim=-1)
    skips = []
    for stage in unet.down_blocks:
        res1, res2, downsample = cast(torch.nn.ModuleList, stage)
        x = res2(res1(x, cond), cond)
        skips.append(x)
        x = downsample(x)
    for block in unet.mid_blocks:
        x = block(x, cond)
    for stage in unet.up_blocks:
        res1, res2, upsample = cast(torch.nn.ModuleList, stage)
        x = torch.cat((x, skips.pop()), dim=1)
        x = res2(res1(x, cond), cond)
        x = upsample(x)
    return unet.head(x).transpose(1, 2)

In [ ]:
# Check: Exercise 13.2
# The learner network has your forward function and the package plan; weights come from the pinned network.
learner = unet_from_lerobot(PINNED, forward_fn=unet_forward, plan_fn=ref.unet_channel_plan)
out = learner(X, T_STEPS, COND)
assert out.shape == X.shape, "Return (B, H, A)."
close(out, PINNED(X, T_STEPS, COND), "output differs from the pinned U-Net")
x1 = X.clone().requires_grad_(True)
learner(x1, T_STEPS, COND).square().sum().backward()
x2 = X.clone().requires_grad_(True)
PINNED(x2, T_STEPS, COND).square().sum().backward()
close(x1.grad, x2.grad, "input gradient", atol=1e-4)
close(learner.head[1].weight.grad, PINNED.final_conv[1].weight.grad, "head gradient", atol=1e-4)
close(
    learner.down_blocks[0][0].conv1.block[0].weight.grad,
    PINNED.down_modules[0][0].conv1.block[0].weight.grad,
    "first-block gradient (flows through the skip)",
    atol=1e-4,
)
close(learner(X, T_STEPS, COND), out, "repeatable")
no_cond = unet_from_lerobot(
    pinned_unet((16, 32), cond_dim=0), forward_fn=unet_forward, plan_fn=ref.unet_channel_plan
)
close(
    no_cond(X, T_STEPS, None),
    pinned_unet((16, 32), cond_dim=0)(X, T_STEPS, None),
    "network without a global condition",
    atol=1e-5,
)
print("Exercise 13.2 passed")

# More behaviour checks: lesson13_checks.py, next to this notebook.
from lesson13_checks import check_unet_forward

check_unet_forward(unet_forward)